In [1]:
import numpy as np
import panel as pn
import plotly.graph_objects as go

pn.extension("plotly")

# ============================================================
# APP INTERACTIVA EN PANEL + PLOTLY
# MODELO DE KURAMOTO EN UNA RED GENERAL + PROYECCION AL TORO
# ============================================================

# ------------------------------------------------------------
# UTILIDADES
# ------------------------------------------------------------
TWO_PI = 2.0 * np.pi


def deg2rad(deg):
    return np.asarray(deg, dtype=float) * np.pi / 180.0


def rad2deg(rad):
    return np.asarray(rad, dtype=float) * 180.0 / np.pi


def wrap_to_2pi(angle):
    return np.mod(angle, TWO_PI)


def torus_embedding(theta1, theta2, R=3.0, r=1.0):
    x = (R + r * np.cos(theta2)) * np.cos(theta1)
    y = (R + r * np.cos(theta2)) * np.sin(theta1)
    z = r * np.sin(theta2)
    return x, y, z


@pn.cache(max_items=32)
def make_torus_mesh(R=3.0, r=1.0, nu=70, nv=40):
    u = np.linspace(0, TWO_PI, nu)
    v = np.linspace(0, TWO_PI, nv)
    U, V = np.meshgrid(u, v)
    X = (R + r * np.cos(V)) * np.cos(U)
    Y = (R + r * np.cos(V)) * np.sin(U)
    Z = r * np.sin(V)
    return X, Y, Z


# ------------------------------------------------------------
# GRAFOS
# ------------------------------------------------------------
def cycle_adjacency(n):
    A = np.zeros((n, n), dtype=float)
    idx = np.arange(n)
    A[idx, (idx - 1) % n] = 1.0
    A[idx, (idx + 1) % n] = 1.0
    return A


def path_adjacency(n):
    A = np.zeros((n, n), dtype=float)
    idx = np.arange(n - 1)
    A[idx, idx + 1] = 1.0
    A[idx + 1, idx] = 1.0
    return A


def complete_adjacency(n):
    return np.ones((n, n), dtype=float) - np.eye(n)


def star_adjacency(n):
    A = np.zeros((n, n), dtype=float)
    if n > 1:
        A[0, 1:] = 1.0
        A[1:, 0] = 1.0
    return A


def grid_adjacency(rows, cols):
    n = rows * cols
    A = np.zeros((n, n), dtype=float)

    def idx(r, c):
        return r * cols + c

    for r in range(rows):
        for c in range(cols):
            i = idx(r, c)
            for rr, cc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
                if 0 <= rr < rows and 0 <= cc < cols:
                    j = idx(rr, cc)
                    A[i, j] = 1.0
    return A


def graph_layout(graph_type, n):
    if graph_type in ("cycle", "complete"):
        angles = np.linspace(0, TWO_PI, n, endpoint=False)
        return np.column_stack((np.cos(angles), np.sin(angles)))

    if graph_type == "path":
        x = np.linspace(-1, 1, n)
        y = np.zeros(n)
        return np.column_stack((x, y))

    if graph_type == "star":
        if n == 1:
            return np.array([[0.0, 0.0]])
        angles = np.linspace(0, TWO_PI, n - 1, endpoint=False)
        pts = np.vstack(([0.0, 0.0], np.column_stack((np.cos(angles), np.sin(angles)))))
        return pts

    if graph_type == "grid":
        rows = int(np.floor(np.sqrt(n)))
        cols = int(np.ceil(n / rows))
        pts = []
        for r in range(rows):
            for c in range(cols):
                if len(pts) < n:
                    pts.append([c, -r])
        pts = np.array(pts, dtype=float)
        if len(pts):
            pts[:, 0] -= np.mean(pts[:, 0])
            pts[:, 1] -= np.mean(pts[:, 1])
            scale = max(np.max(np.abs(pts[:, 0])), np.max(np.abs(pts[:, 1])), 1.0)
            pts /= scale
        return pts

    angles = np.linspace(0, TWO_PI, n, endpoint=False)
    return np.column_stack((np.cos(angles), np.sin(angles)))


@pn.cache(max_items=64)
def build_graph(graph_type, n):
    if graph_type == "cycle":
        A = cycle_adjacency(n)
    elif graph_type == "path":
        A = path_adjacency(n)
    elif graph_type == "complete":
        A = complete_adjacency(n)
    elif graph_type == "star":
        A = star_adjacency(n)
    elif graph_type == "grid":
        rows = int(np.floor(np.sqrt(n)))
        cols = int(np.ceil(n / rows))
        A_big = grid_adjacency(rows, cols)
        A = A_big[:n, :n]
    else:
        A = cycle_adjacency(n)

    degrees = np.sum(A, axis=1)
    L = np.diag(degrees) - A
    pos = graph_layout(graph_type, n)
    evals = np.linalg.eigvalsh(L)
    return A, L, pos, evals


# ------------------------------------------------------------
# CONDICIONES INICIALES Y FRECUENCIAS
# ------------------------------------------------------------
def generate_initial_condition(n, mode, seed=1):
    rng = np.random.default_rng(seed)

    if mode == "rampa":
        return np.linspace(20, 340, n)

    if mode == "aleatoria":
        return rng.uniform(0, 360, n)

    if mode == "dos grupos":
        a = np.full(n // 2, 40.0)
        b = np.full(n - n // 2, 240.0)
        return np.concatenate((a, b))

    if mode == "senoidal":
        k = np.arange(n)
        vals = 180 + 140 * np.sin(2 * np.pi * k / max(n, 2))
        return np.mod(vals, 360)

    return np.linspace(20, 340, n)


def generate_omega(n, mode, scale=1.0, seed=1):
    rng = np.random.default_rng(seed + 1000)

    if mode == "iguales":
        return np.zeros(n)

    if mode == "rampa":
        return scale * np.linspace(-1.0, 1.0, n)

    if mode == "aleatorias":
        return scale * rng.normal(0.0, 1.0, n)

    if mode == "dos grupos":
        a = np.full(n // 2, -scale)
        b = np.full(n - n // 2, scale)
        return np.concatenate((a, b))

    return np.zeros(n)


# ------------------------------------------------------------
# DINAMICA DE KURAMOTO
# ------------------------------------------------------------
def kuramoto_rhs(theta, A, omega, K):
    diff = theta[np.newaxis, :] - theta[:, np.newaxis]
    return omega + K * np.sum(A * np.sin(diff), axis=1)


def simulate_kuramoto_general(theta0_deg, A, omega, K, T, dt):
    theta0 = deg2rad(theta0_deg)
    omega = np.asarray(omega, dtype=float)

    n_steps = int(np.round(T / dt)) + 1
    t = np.linspace(0.0, T, n_steps)
    Theta = np.zeros((n_steps, len(theta0)), dtype=float)
    Theta[0] = theta0

    for k in range(n_steps - 1):
        y = Theta[k]
        k1 = kuramoto_rhs(y, A, omega, K)
        k2 = kuramoto_rhs(y + 0.5 * dt * k1, A, omega, K)
        k3 = kuramoto_rhs(y + 0.5 * dt * k2, A, omega, K)
        k4 = kuramoto_rhs(y + dt * k3, A, omega, K)
        Theta[k + 1] = y + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    Theta_mod = wrap_to_2pi(Theta)
    Theta_deg_mod = rad2deg(Theta_mod)
    Theta_deg_unwrapped = rad2deg(Theta)

    z = np.mean(np.exp(1j * Theta), axis=1)
    order_r = np.abs(z)
    order_psi = np.angle(z)

    spread = np.max(Theta, axis=1) - np.min(Theta, axis=1)

    return {
        "t": t,
        "Theta_rad": Theta,
        "Theta_deg_mod": Theta_deg_mod,
        "Theta_deg_unwrapped": Theta_deg_unwrapped,
        "order_r": order_r,
        "order_psi": order_psi,
        "spread_deg": rad2deg(spread),
    }


@pn.cache(max_items=128)
def get_simulation_bundle(graph_type, n, K, T, dt, init_mode, omega_mode, omega_scale, seed):
    A, L, pos, evals = build_graph(graph_type, n)
    theta0_deg = generate_initial_condition(n, init_mode, seed)
    omega = generate_omega(n, omega_mode, omega_scale, seed)
    data = simulate_kuramoto_general(theta0_deg, A, omega, K, T, dt)

    return {
        "A": A,
        "L": L,
        "pos": pos,
        "evals": evals,
        "theta0_deg": theta0_deg,
        "omega": omega,
        "data": data,
    }


# ------------------------------------------------------------
# FIGURAS
# ------------------------------------------------------------
def make_network_figure(data, idx, A, pos):
    xmod = data["Theta_deg_mod"][idx]
    theta = deg2rad(xmod)
    n = A.shape[0]

    edge_i, edge_j = np.where(np.triu(A, 1) != 0)
    edge_x = []
    edge_y = []
    for i, j in zip(edge_i, edge_j):
        edge_x.extend([pos[i, 0], pos[j, 0], None])
        edge_y.extend([pos[i, 1], pos[j, 1], None])

    node_text = [
        f"nodo {i+1}<br>theta{i+1} mod 360 = {xmod[i]:.2f}°"
        for i in range(n)
    ]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=1.5),
        hoverinfo="skip",
        name="Aristas",
    ))
    fig.add_trace(go.Scatter(
        x=pos[:, 0],
        y=pos[:, 1],
        mode="markers+text",
        text=[str(i + 1) for i in range(n)],
        textposition="top center",
        marker=dict(
            size=18,
            color=theta,
            colorscale="HSV",
            cmin=0,
            cmax=TWO_PI,
            colorbar=dict(title="fase"),
        ),
        hovertext=node_text,
        hoverinfo="text",
        name="Nodos",
    ))
    fig.update_layout(
        title="Red general: color = fase modulo 2π",
        width=720,
        height=500,
        margin=dict(l=20, r=20, t=50, b=20),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
        showlegend=False,
    )
    return fig


def make_torus_pair_figure(data, idx, node_i, node_j, trail, R_major, r_minor):
    Xi = deg2rad(data["Theta_deg_mod"][:, node_i])
    Xj = deg2rad(data["Theta_deg_mod"][:, node_j])
    Xp, Yp, Zp = torus_embedding(Xi, Xj, R=R_major, r=r_minor)
    X, Y, Z = make_torus_mesh(R=R_major, r=r_minor)

    start = max(0, idx - trail)

    fig = go.Figure()
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, opacity=0.25, showscale=False, name="Toro"))
    fig.add_trace(go.Scatter3d(
        x=Xp[start:idx + 1],
        y=Yp[start:idx + 1],
        z=Zp[start:idx + 1],
        mode="lines",
        name="Trayectoria",
        line=dict(width=6),
    ))
    fig.add_trace(go.Scatter3d(
        x=[Xp[idx]],
        y=[Yp[idx]],
        z=[Zp[idx]],
        mode="markers",
        name="Estado actual",
        marker=dict(size=6),
    ))
    fig.update_layout(
        title=f"Proyeccion al toro T² de los nodos ({node_i+1}, {node_j+1})",
        width=720,
        height=650,
        margin=dict(l=10, r=10, t=50, b=10),
        scene=dict(aspectmode="data"),
    )
    return fig


def make_states_figure(data, idx):
    t = data["t"]
    Xdeg = data["Theta_deg_unwrapped"]
    n = Xdeg.shape[1]

    fig = go.Figure()
    for i in range(n):
        fig.add_trace(go.Scatter(
            x=t[:idx + 1],
            y=Xdeg[:idx + 1, i],
            mode="lines",
            name=f"theta{i+1}(t)",
        ))

    fig.update_layout(
        title="Fases desenrolladas en R^n",
        xaxis_title="t",
        yaxis_title="Grados",
        width=720,
        height=320,
        margin=dict(l=40, r=20, t=50, b=40),
    )
    return fig


def make_sync_figure(data, idx):
    t = data["t"]
    r = data["order_r"]
    spread = data["spread_deg"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t[:idx + 1], y=r[:idx + 1], mode="lines", name="R(t)"))
    fig.add_trace(go.Scatter(
        x=t[:idx + 1],
        y=spread[:idx + 1],
        mode="lines",
        name="spread (deg)",
        yaxis="y2",
    ))
    fig.update_layout(
        title="Medidas de sincronizacion en Kuramoto",
        xaxis_title="t",
        yaxis=dict(title="R(t)", range=[-0.02, 1.02]),
        yaxis2=dict(title="spread (deg)", overlaying="y", side="right"),
        width=720,
        height=320,
        margin=dict(l=40, r=40, t=50, b=40),
    )
    return fig


def make_spectrum_figure(evals):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=np.arange(1, len(evals) + 1),
        y=evals,
        name="Autovalores",
    ))
    fig.update_layout(
        title="Espectro de la matriz Laplaciana L = D - A del grafo",
        xaxis_title="Indice",
        yaxis_title="Autovalor",
        width=720,
        height=260,
        margin=dict(l=40, r=20, t=50, b=40),
    )
    return fig


def make_plane_figure(data, idx, node_i, node_j, trail=300):
    Xdeg = data["Theta_deg_unwrapped"]
    xi = Xdeg[:, node_i]
    xj = Xdeg[:, node_j]

    start = max(0, idx - trail)
    minv = min(np.min(xi[:idx + 1]), np.min(xj[:idx + 1]))
    maxv = max(np.max(xi[:idx + 1]), np.max(xj[:idx + 1]))
    pad = 20

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=xi[start:idx + 1],
        y=xj[start:idx + 1],
        mode="lines",
        name=f"Trayectoria (θ{node_i+1}, θ{node_j+1})",
    ))
    fig.add_trace(go.Scatter(
        x=[xi[idx]],
        y=[xj[idx]],
        mode="markers",
        name="Estado actual",
        marker=dict(size=10),
    ))
    fig.add_trace(go.Scatter(
        x=[xi[0]],
        y=[xj[0]],
        mode="markers",
        name="Estado inicial",
        marker=dict(size=9, symbol="diamond"),
    ))
    fig.add_trace(go.Scatter(
        x=[minv - pad, maxv + pad],
        y=[minv - pad, maxv + pad],
        mode="lines",
        name="Diagonal θ_i = θ_j",
        line=dict(dash="dash"),
    ))
    fig.update_layout(
        title=f"Espacio de estados y diagonal de sincronía: (θ{node_i+1}, θ{node_j+1})",
        xaxis_title=f"θ{node_i+1} (grados desenrollados)",
        yaxis_title=f"θ{node_j+1} (grados desenrollados)",
        width=720,
        height=450,
        margin=dict(l=40, r=20, t=50, b=40),
    )
    return fig


# ------------------------------------------------------------
# WIDGETS
# ------------------------------------------------------------
graph_widget = pn.widgets.Select(
    name="Tipo de grafo",
    options=["cycle", "path", "complete", "star", "grid"],
    value="cycle",
)

n_widget = pn.widgets.IntSlider(name="Numero de nodos n", start=2, end=20, step=1, value=5)
K_widget = pn.widgets.FloatSlider(name="Acoplamiento K", start=0.0, end=20.0, step=0.05, value=0.5)

omega_mode_widget = pn.widgets.Select(
    name="Frecuencias naturales",
    options=["iguales", "rampa", "aleatorias", "dos grupos"],
    value="rampa",
)
omega_scale_widget = pn.widgets.FloatSlider(name="Escala de omega", start=0.0, end=3.0, step=0.05, value=1.0)

init_mode_widget = pn.widgets.Select(
    name="Condicion inicial",
    options=["rampa", "aleatoria", "dos grupos", "senoidal"],
    value="rampa",
)

T_widget = pn.widgets.FloatSlider(name="Tiempo total T", start=2, end=80, step=1, value=20)
dt_widget = pn.widgets.FloatSlider(name="dt", start=0.005, end=0.1, step=0.005, value=0.02)
trail_widget = pn.widgets.IntSlider(name="Cola visible en toro", start=5, end=500, step=5, value=120)
R_widget = pn.widgets.FloatSlider(name="Radio mayor del toro", start=1.5, end=5.0, step=0.1, value=3.0)
r_widget = pn.widgets.FloatSlider(name="Radio menor del toro", start=0.3, end=2.0, step=0.05, value=1.0)

pair_i_widget = pn.widgets.IntSlider(name="Nodo i para toro", start=1, end=2, step=1, value=1)
pair_j_widget = pn.widgets.IntSlider(name="Nodo j para toro", start=1, end=2, step=1, value=2)

play_widget = pn.widgets.Player(
    name="Tiempo",
    start=0,
    end=100,
    step=1,
    interval=80,
    value=0,
    loop_policy="loop",
)

reset_btn = pn.widgets.Button(name="Restablecer parametros", button_type="primary")
randomize_btn = pn.widgets.Button(name="Nueva condicion aleatoria", button_type="success")
seed_widget = pn.widgets.IntInput(name="Semilla", value=1)


# ------------------------------------------------------------
# EVENTOS
# ------------------------------------------------------------
def valid_pair_indices(n, pair_i, pair_j):
    i = max(0, min(n - 1, pair_i - 1))
    j = max(0, min(n - 1, pair_j - 1))
    if i == j:
        j = (j + 1) % n
    return i, j


def update_pair_bounds(event=None):
    n = n_widget.value
    pair_i_widget.end = n
    pair_j_widget.end = n

    if pair_i_widget.value > n:
        pair_i_widget.value = n
    if pair_j_widget.value > n:
        pair_j_widget.value = n

    if pair_i_widget.value == pair_j_widget.value:
        pair_j_widget.value = min(n, pair_i_widget.value + 1) if pair_i_widget.value < n else max(1, pair_i_widget.value - 1)


def refresh_random_condition(event=None):
    # Cambiamos semilla para forzar nueva realizacion
    seed_widget.value = int(np.random.randint(1, 10**6))
    play_widget.value = 0


def reset_params(event=None):
    graph_widget.value = "cycle"
    n_widget.value = 5
    K_widget.value = 0.5
    omega_mode_widget.value = "rampa"
    omega_scale_widget.value = 1.0
    init_mode_widget.value = "rampa"
    T_widget.value = 20
    dt_widget.value = 0.02
    trail_widget.value = 120
    R_widget.value = 3.0
    r_widget.value = 1.0
    pair_i_widget.value = 1
    pair_j_widget.value = 2
    seed_widget.value = 1
    play_widget.value = 0


def sync_player_bounds(event=None):
    bundle = get_simulation_bundle(
        graph_widget.value,
        n_widget.value,
        K_widget.value,
        T_widget.value,
        dt_widget.value,
        init_mode_widget.value,
        omega_mode_widget.value,
        omega_scale_widget.value,
        seed_widget.value,
    )
    n_steps = len(bundle["data"]["t"])
    play_widget.end = max(0, n_steps - 1)
    if play_widget.value > play_widget.end:
        play_widget.value = play_widget.end


randomize_btn.on_click(refresh_random_condition)
reset_btn.on_click(reset_params)

n_widget.param.watch(update_pair_bounds, "value")

for w in [
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget, seed_widget
]:
    w.param.watch(sync_player_bounds, "value")

update_pair_bounds()
sync_player_bounds()


# ------------------------------------------------------------
# PANELES REACTIVOS
# ------------------------------------------------------------
def get_bundle():
    return get_simulation_bundle(
        graph_widget.value,
        n_widget.value,
        K_widget.value,
        T_widget.value,
        dt_widget.value,
        init_mode_widget.value,
        omega_mode_widget.value,
        omega_scale_widget.value,
        seed_widget.value,
    )


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget, play_widget
)
def status_panel(*_):
    bundle = get_bundle()
    data = bundle["data"]
    evals = bundle["evals"]
    omega = bundle["omega"]

    idx = min(play_widget.value, len(data["t"]) - 1)
    xmod = data["Theta_deg_mod"][idx]
    mean_mod = np.mean(xmod)

    return pn.pane.Markdown(
        f"""### Estado actual
- **Grafo** = {graph_widget.value}
- **n** = {n_widget.value}
- **t** = {data["t"][idx]:.2f}
- **K** = {K_widget.value:.2f}
- **R(t)** = {data["order_r"][idx]:.4f}
- **spread** = {data["spread_deg"][idx]:.4f}°
- **autovalor 0 de L** = {evals[0]:.6f}
- **segundo autovalor λ2 de L** = {evals[1]:.6f}

### Teoria
- **dtheta_i/dt = omega_i + K sum_j A_ij sin(theta_j - theta_i)**
- **R(t)** cerca de 1 indica sincronizacion alta.
- El espectro de **L = D - A** describe la conectividad estructural del grafo.

### Estado medio actual
- promedio modulo 360 ≈ {mean_mod:.2f}°
- omega promedio = {np.mean(omega):.4f}
- ejemplo nodo 1 = {xmod[0]:.2f}°
"""
    )


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget
)
def initial_condition_panel(*_):
    bundle = get_bundle()
    vals = bundle["theta0_deg"]
    ome = bundle["omega"]

    txt1 = "\n".join([f"- theta{i+1}(0) = {vals[i]:.2f}°" for i in range(len(vals))])
    txt2 = "\n".join([f"- omega{i+1} = {ome[i]:.4f}" for i in range(len(ome))])

    return pn.pane.Markdown(
        f"""### Condiciones iniciales actuales
{txt1}

### Frecuencias naturales actuales
{txt2}
"""
    )


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget, play_widget
)
def network_panel(*_):
    bundle = get_bundle()
    data = bundle["data"]
    idx = min(play_widget.value, len(data["t"]) - 1)
    return pn.pane.Plotly(
        make_network_figure(data, idx, bundle["A"], bundle["pos"]),
        config={"responsive": True},
    )


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget, trail_widget, R_widget, r_widget,
    pair_i_widget, pair_j_widget, play_widget
)
def torus_panel(*_):
    bundle = get_bundle()
    data = bundle["data"]
    idx = min(play_widget.value, len(data["t"]) - 1)
    i, j = valid_pair_indices(n_widget.value, pair_i_widget.value, pair_j_widget.value)

    return pn.pane.Plotly(
        make_torus_pair_figure(
            data, idx, i, j, trail_widget.value, R_widget.value, r_widget.value
        ),
        config={"responsive": True},
    )


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget, play_widget
)
def states_panel(*_):
    bundle = get_bundle()
    data = bundle["data"]
    idx = min(play_widget.value, len(data["t"]) - 1)
    return pn.pane.Plotly(make_states_figure(data, idx), config={"responsive": True})


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget, play_widget
)
def sync_panel(*_):
    bundle = get_bundle()
    data = bundle["data"]
    idx = min(play_widget.value, len(data["t"]) - 1)
    return pn.pane.Plotly(make_sync_figure(data, idx), config={"responsive": True})


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget
)
def spectrum_panel(*_):
    bundle = get_bundle()
    return pn.pane.Plotly(make_spectrum_figure(bundle["evals"]), config={"responsive": True})


@pn.depends(
    graph_widget, n_widget, K_widget, T_widget, dt_widget,
    init_mode_widget, omega_mode_widget, omega_scale_widget,
    seed_widget, pair_i_widget, pair_j_widget, play_widget
)
def plane_panel(*_):
    bundle = get_bundle()
    data = bundle["data"]
    idx = min(play_widget.value, len(data["t"]) - 1)
    i, j = valid_pair_indices(n_widget.value, pair_i_widget.value, pair_j_widget.value)

    return pn.pane.Plotly(
        make_plane_figure(data, idx, i, j),
        config={"responsive": True},
    )


# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------
controls = pn.Card(
    "Elige un grafo arbitrario y observa el modelo de Kuramoto sobre la red, el toro y las medidas de sincronizacion.",
    graph_widget,
    n_widget,
    K_widget,
    omega_mode_widget,
    omega_scale_widget,
    init_mode_widget,
    seed_widget,
    pn.Row(randomize_btn, reset_btn),
    T_widget,
    dt_widget,
    trail_widget,
    R_widget,
    r_widget,
    pair_i_widget,
    pair_j_widget,
    play_widget,
    title="Controles",
    width=380,
)

main = pn.Column(
    pn.pane.Markdown("## Modelo de Kuramoto en una red general y proyeccion al toro"),
    pn.pane.Markdown(
        "Esta app repite el ejercicio del flujo laplaciano, pero ahora con el modelo de **Kuramoto**. "
        "Puedes ver al mismo tiempo la red, la proyeccion al toro, las fases desenrolladas, "
        "las medidas de sincronizacion y el espectro de la matriz Laplaciana estructural del grafo."
    ),
    pn.Row(
        controls,
        pn.Column(
            status_panel,
            initial_condition_panel,
            network_panel,
            torus_panel,
            states_panel,
            plane_panel,
            sync_panel,
            spectrum_panel,
        ),
    ),
)

main.servable()
main

Column
    [0] Markdown(str)
    [1] Markdown(str)
    [2] Row
        [0] Card(title='Controles', width=380)
            [0] Markdown(str)
            [1] Select(name='Tipo de grafo', options=['cycle', 'path', ...], value='cycle')
            [2] IntSlider(end=20, name='Numero de nodos n', start=2, value=5)
            [3] FloatSlider(end=20.0, name='Acoplamiento K', step=0.05, value=0.5)
            [4] Select(name='Frecuencias naturales', options=['iguales', 'rampa', ...], value='rampa')
            [5] FloatSlider(end=3.0, name='Escala de omega', step=0.05, value=1.0)
            [6] Select(name='Condicion inicial', options=['rampa', 'aleatoria', ...], value='rampa')
            [7] IntInput(name='Semilla', value=1)
            [8] Row
                [0] Button(button_type='success', name='Nueva condicion a...)
                [1] Button(button_type='primary', name='Restablecer parametros')
            [9] FloatSlider(end=80, name='Tiempo total T', start=2, step=1, value=20)
            [10] FloatSlider(end=0.1, name='dt', start=0.005, step=0.005, value=0.02)
            [11] IntSlider(end=500, name='Cola visible en toro', start=5, step=5, value=120)
            [12] FloatSlider(end=5.0, name='Radio mayor del toro', start=1.5, value=3.0)
            [13] FloatSlider(end=2.0, name='Radio menor del toro', start=0.3, step=0.05, value=1.0)
            [14] IntSlider(end=5, name='Nodo i para toro', start=1, value=1)
            [15] IntSlider(end=5, name='Nodo j para toro', start=1, value=2)
            [16] Player(end=1000, interval=80, loop_policy='loop', name='Tiempo')
        [1] Column
            [0] ParamFunction(function, _pane=Markdown, defer_load=False)
            [1] ParamFunction(function, _pane=Markdown, defer_load=False)
            [2] ParamFunction(function, _pane=Plotly, defer_load=False)
            [3] ParamFunction(function, _pane=Plotly, defer_load=False)
            [4] ParamFunction(function, _pane=Plotly, defer_load=False)
            [5] ParamFunction(function, _pane=Plotly, defer_load=False)
            [6] ParamFunction(function, _pane=Plotly, defer_load=False)
            [7] ParamFunction(function, _pane=Plotly, defer_load=False)